# 🛒 Know Your Customer — Smart Clustering using Unsupervised Learning

[![LinkedIn](https://img.shields.io/badge/LinkedIn-Rohit%20Kaushal-blue?style=flat&logo=linkedin)](https://www.linkedin.com/in/kaushal-rohit-83911a1b6)

**Author:** Rohit Kaushal Bharatbhai | KPGU Vadodara

---

## 📌 Project Overview

This notebook segments retail customers into distinct behavioral groups using **K-Means Clustering**.
The pipeline covers:
- Data cleaning & feature engineering
- Dimensionality reduction with PCA
- Optimal cluster selection (Elbow + Silhouette)
- Cluster profiling & business recommendations

## 1. 📦 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All libraries loaded successfully!")

## 2. 📂 Data Loading & Initial Preprocessing

We load the SmartCart customer dataset and handle missing values immediately.
The `Income` column has some nulls — we fill them with the **median** to avoid skewing the distribution.

In [ ]:
c = pd.read_csv("smartcart_customers.csv")

# Handle missing values
c["Income"] = c["Income"].fillna(c["Income"].median())

print(f"Dataset shape: {c.shape}")
print(f"\nMissing values after fill:\n{c.isnull().sum()[c.isnull().sum() > 0]}")
c.head()

## 3. 🛠️ Feature Engineering

We derive several meaningful features from the raw columns:
- **Age** from birth year
- **JoinDate** (days since joining) from `Dt_Customer`
- **Total_Spending** by summing all product category spend columns
- **Avg_Spending_Per_Month** to normalize spending over tenure
- **Total_Children** combining kids and teens
- **Education** and **Living_With** standardized into cleaner categories

In [ ]:
# Age
c["Age"] = 2026 - c["Year_Birth"]

# Customer tenure in days
c["Dt_Customer"] = pd.to_datetime(c["Dt_Customer"], dayfirst=True)
reference = c["Dt_Customer"].max()
c["JoinDate"] = (reference - c["Dt_Customer"]).dt.days

# Spending aggregation
spending_cols = ["MntWines", "MntFruits", "MntMeatProducts",
                 "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
c["Total_Spending"] = c[spending_cols].sum(axis=1)
c["Avg_Spending_Per_Month"] = c["Total_Spending"] / (c["JoinDate"] + 1)

# Children
c["Total_Children"] = c["Kidhome"] + c["Teenhome"]

# Education standardization
c["Education"] = c["Education"].replace({
    "Basic": "Undergraduate", "2n Cycle": "Undergraduate",
    "Graduation": "Graduate",
    "Master": "Postgraduate", "PhD": "Postgraduate"
})

# Living situation
c["Living_With"] = c["Marital_Status"].replace({
    "Married": "Partner", "Together": "Partner",
    "Single": "Alone", "Divorced": "Alone",
    "Widow": "Alone", "Absurd": "Alone", "YOLO": "Alone"
})

# Additional features
c["Response_Rate"] = (c["Response"] / (c["NumDealsPurchases"] + 1)) * 100
c["Web_Purchase_Ratio"] = c["NumWebPurchases"] / (
    c["NumWebPurchases"] + c["NumCatalogPurchases"] + c["NumStorePurchases"] + 1
)

print("✅ Feature engineering complete!")
print(f"New features: Age, JoinDate, Total_Spending, Avg_Spending_Per_Month, Total_Children")
c[["Age", "JoinDate", "Total_Spending", "Avg_Spending_Per_Month", "Total_Children"]].describe()

## 4. 🧹 Outlier Removal

We remove obvious data quality issues:
- Ages outside the realistic range of 18–90
- Incomes above $600,000 (likely data errors)
- More than 4 children (implausible for this dataset)

In [ ]:
print(f"Initial data size: {len(c)}")

c_cleaned = c[
    (c["Age"] < 90) &
    (c["Age"] > 18) &
    (c["Income"] < 600_000) &
    (c["Total_Children"] < 5)
].copy()

print(f"After outlier removal: {len(c_cleaned)}")
print(f"Records removed: {len(c) - len(c_cleaned)}")

## 5. 🔢 Feature Selection & Encoding

We select a mix of numerical and categorical features. Categorical features (`Education`, `Living_With`) are **one-hot encoded** so the model treats them numerically without any ordinal assumption.

In [ ]:
feature_cols = [
    "Income", "Recency", "Response", "Age", "Total_Spending",
    "Total_Children", "Avg_Spending_Per_Month", "Response_Rate",
    "Web_Purchase_Ratio", "NumWebPurchases", "NumCatalogPurchases"
]

cat_cols = ["Education", "Living_With"]
ohe = OneHotEncoder(sparse_output=False)
enc_cols = ohe.fit_transform(c_cleaned[cat_cols])
enc_df = pd.DataFrame(
    enc_cols,
    columns=ohe.get_feature_names_out(cat_cols),
    index=c_cleaned.index
)

c_encoded = pd.concat([c_cleaned[feature_cols], enc_df], axis=1)

print(f"Total features used for clustering: {c_encoded.shape[1]}")
print(f"\nFeature list:")
for f in c_encoded.columns:
    print(f"  - {f}")

## 6. 📏 Standardization & PCA

K-Means is **distance-based**, so feature scaling is essential. We apply `StandardScaler` to bring all features to the same scale.

PCA (3 components) is applied **only for visualization** purposes — the actual clustering happens on the full scaled feature space.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(c_encoded)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print("PCA RESULTS")
print("-" * 40)
print(f"Component 1 variance: {pca.explained_variance_ratio_[0]:.2%}")
print(f"Component 2 variance: {pca.explained_variance_ratio_[1]:.2%}")
print(f"Component 3 variance: {pca.explained_variance_ratio_[2]:.2%}")
print(f"Total variance (3 PCs): {pca.explained_variance_ratio_.sum():.2%}")

## 7. 📊 Finding Optimal K — Elbow Method & Silhouette Score

We test K values from 2 to 10:
- **Elbow Method:** Look for the "elbow" where inertia stops dropping sharply
- **Silhouette Score:** Higher is better; measures how well-separated clusters are

In [ ]:
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))
    print(f"K={k:2d} | Inertia: {kmeans.inertia_:,.0f} | Silhouette: {silhouette_scores[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method For Optimal K', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score For Optimal K', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('optimal_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"\n✅ Optimal K = {optimal_k} | Best Silhouette Score = {max(silhouette_scores):.4f}")

## 8. 🎯 Final K-Means Clustering

With optimal K determined, we run the final K-Means model and assign each customer a cluster label.

In [ ]:
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
c_cleaned['Cluster'] = kmeans_final.fit_predict(X_scaled)

print(f"K-Means applied with K={optimal_k}")
print(f"\nCluster distribution:")
dist = c_cleaned['Cluster'].value_counts().sort_index()
for cluster, count in dist.items():
    pct = count / len(c_cleaned) * 100
    bar = '█' * int(pct / 1)
    print(f"  Cluster {cluster:2d}: {count:4d} customers ({pct:.1f}%) {bar}")

## 9. 🌐 3D PCA Visualization of Clusters

We project the clusters into 3D PCA space for visual inspection. Each color represents one cluster.

In [ ]:
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

colors = plt.cm.Set3(np.linspace(0, 1, optimal_k))

for cluster in range(optimal_k):
    cluster_data = X_pca[c_cleaned['Cluster'] == cluster]
    ax.scatter(
        cluster_data[:, 0],
        cluster_data[:, 1],
        cluster_data[:, 2],
        c=[colors[cluster]],
        label=f'Cluster {cluster}',
        s=100,
        alpha=0.6,
        edgecolors='black',
        linewidth=0.5
    )

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]:.1%})', fontsize=11)
ax.set_title('Customer Segments in PCA Space', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)

plt.savefig('clusters_3d_pca.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. 🔍 Detailed Cluster Behavioral Analysis

For each cluster, we examine demographics, spending behaviour, engagement metrics, and family composition.

In [ ]:
analysis_features = [
    'Income', 'Age', 'Total_Spending', 'Total_Children',
    'Recency', 'Response', 'JoinDate', 'Avg_Spending_Per_Month',
    'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases'
]

cluster_analysis = pd.DataFrame()

for cluster in sorted(c_cleaned['Cluster'].unique()):
    cluster_data = c_cleaned[c_cleaned['Cluster'] == cluster]

    print(f"\n{'=' * 65}")
    print(f" CLUSTER {cluster} PROFILE  ({len(cluster_data)} customers | {len(cluster_data)/len(c_cleaned)*100:.1f}%)")
    print(f"{'=' * 65}")
    print(f"  Age:      {cluster_data['Age'].mean():.1f} yrs  |  Income: ${cluster_data['Income'].mean():,.0f}  |  Children: {cluster_data['Total_Children'].mean():.2f}")
    print(f"  Spending: ${cluster_data['Total_Spending'].mean():,.0f} total  |  ${cluster_data['Avg_Spending_Per_Month'].mean():.2f}/month")
    print(f"  Wines: ${cluster_data['MntWines'].mean():,.0f}  |  Meat: ${cluster_data['MntMeatProducts'].mean():,.0f}  |  Fish: ${cluster_data['MntFishProducts'].mean():,.0f}")
    print(f"  Response: {cluster_data['Response'].mean()*100:.1f}%  |  Recency: {cluster_data['Recency'].mean():.0f} days  |  Tenure: {cluster_data['JoinDate'].mean():.0f} days")
    print(f"  Web: {cluster_data['NumWebPurchases'].mean():.1f}  |  Catalog: {cluster_data['NumCatalogPurchases'].mean():.1f}  |  Store: {cluster_data['NumStorePurchases'].mean():.1f}")
    edu = cluster_data['Education'].value_counts().to_dict()
    living = cluster_data['Living_With'].value_counts().to_dict()
    print(f"  Education: {edu}")
    print(f"  Living:    {living}")

    cluster_analysis = pd.concat([
        cluster_analysis,
        pd.DataFrame({
            'Cluster': [cluster],
            'Size': [len(cluster_data)],
            'Avg_Income': [cluster_data['Income'].mean()],
            'Avg_Age': [cluster_data['Age'].mean()],
            'Total_Spending': [cluster_data['Total_Spending'].mean()],
            'Response_Rate': [cluster_data['Response'].mean()],
            'Recency': [cluster_data['Recency'].mean()],
            'Monthly_Spending': [cluster_data['Avg_Spending_Per_Month'].mean()]
        })
    ], ignore_index=True)

## 11. 📊 Comparative Cluster Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
colors_bar = plt.cm.Set3(np.linspace(0, 1, optimal_k))

axes[0, 0].bar(cluster_analysis['Cluster'], cluster_analysis['Avg_Income'], color=colors_bar)
axes[0, 0].set_title('Average Income by Cluster', fontweight='bold')
axes[0, 0].set_ylabel('Income ($)')
axes[0, 0].set_xlabel('Cluster')

axes[0, 1].bar(cluster_analysis['Cluster'], cluster_analysis['Avg_Age'], color=colors_bar)
axes[0, 1].set_title('Average Age by Cluster', fontweight='bold')
axes[0, 1].set_ylabel('Age (years)')
axes[0, 1].set_xlabel('Cluster')

axes[0, 2].bar(cluster_analysis['Cluster'], cluster_analysis['Total_Spending'], color=colors_bar)
axes[0, 2].set_title('Average Total Spending by Cluster', fontweight='bold')
axes[0, 2].set_ylabel('Spending ($)')
axes[0, 2].set_xlabel('Cluster')

axes[1, 0].bar(cluster_analysis['Cluster'], cluster_analysis['Response_Rate']*100, color=colors_bar)
axes[1, 0].set_title('Response Rate by Cluster', fontweight='bold')
axes[1, 0].set_ylabel('Response Rate (%)')
axes[1, 0].set_xlabel('Cluster')

axes[1, 1].bar(cluster_analysis['Cluster'], cluster_analysis['Monthly_Spending'], color=colors_bar)
axes[1, 1].set_title('Average Monthly Spending by Cluster', fontweight='bold')
axes[1, 1].set_ylabel('Monthly Spending ($)')
axes[1, 1].set_xlabel('Cluster')

axes[1, 2].bar(cluster_analysis['Cluster'], cluster_analysis['Size'], color=colors_bar)
axes[1, 2].set_title('Cluster Size Distribution', fontweight='bold')
axes[1, 2].set_ylabel('Number of Customers')
axes[1, 2].set_xlabel('Cluster')

plt.tight_layout()
plt.savefig('cluster_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 12. 💡 Business Insights & Recommendations

Based on the cluster profiles, here are automatically generated actionable insights for each segment.

In [ ]:
print("BUSINESS INSIGHTS & RECOMMENDATIONS")
print("=" * 65)

for idx, row in cluster_analysis.iterrows():
    cluster = int(row['Cluster'])
    cluster_data = c_cleaned[c_cleaned['Cluster'] == cluster]

    print(f"\nCLUSTER {cluster} — Action Items")
    print("-" * 65)

    if row['Avg_Income'] > cluster_analysis['Avg_Income'].quantile(0.75):
        print("  ✓ HIGH-VALUE SEGMENT: Focus on premium products and exclusive offerings")
    elif row['Avg_Income'] < cluster_analysis['Avg_Income'].quantile(0.25):
        print("  ✓ BUDGET SEGMENT: Offer value bundles and loyalty discounts")

    if row['Response_Rate'] > cluster_analysis['Response_Rate'].quantile(0.75):
        print("  ✓ HIGHLY ENGAGED: Leverage for referral programs and VIP benefits")
    elif row['Response_Rate'] < cluster_analysis['Response_Rate'].quantile(0.25):
        print("  ✓ LOW ENGAGEMENT: Re-engagement campaigns with special incentives needed")

    if row['Recency'] > cluster_analysis['Recency'].quantile(0.75):
        print("  ✓ AT-RISK SEGMENT: Immediate win-back campaigns recommended")
    else:
        print("  ✓ LOYAL SEGMENT: Maintain engagement through regular communication")

    if row['Total_Spending'] > cluster_analysis['Total_Spending'].quantile(0.75):
        print("  ✓ HIGH SPENDERS: Cross-sell and upsell opportunities")

    if cluster_data['Total_Children'].mean() > 1:
        print("  ✓ FAMILY-ORIENTED: Family packages and bulk discounts effective")

print("\n" + "=" * 65)
print("Analysis complete! ✅")

---

## 📋 Summary Table

In [ ]:
summary = cluster_analysis.copy()
summary['Response_Rate'] = (summary['Response_Rate'] * 100).round(1).astype(str) + '%'
summary['Avg_Income'] = summary['Avg_Income'].apply(lambda x: f"${x:,.0f}")
summary['Total_Spending'] = summary['Total_Spending'].apply(lambda x: f"${x:,.0f}")
summary['Avg_Age'] = summary['Avg_Age'].round(1)
summary.columns = ['Cluster', 'Size', 'Avg Income', 'Avg Age', 'Total Spending', 'Response Rate', 'Recency (days)', 'Monthly Spending']
summary.set_index('Cluster', inplace=True)
summary

---

**Made   by Rohit Kaushal   
[LinkedIn](https://www.linkedin.com/in/kaushal-rohit-83911a1b6)